# FDS Challenge 2026: final submission notebook — CatBoost only

This notebook reproduces our final submission end to end:

1. Load the training data.
2. Build lookup tables **from the training set only**.
3. Turn each battle into 134 features computed on the 30 observed turns (122 state/dynamics features + 6 pairwise duels
   + 6 features on P2's team when it can be identified), plus 14 identity columns used as categories by CatBoost.
4. Add 3 learned ratings fitted on training labels only: team strength, species strength, and a "battle token" rating
   (which Pokémon are still alive and how healthy, their status, the moves used).
5. Validate the final pipeline with 5-fold stratified cross-validation.
6. Train on the full training set, load `test.jsonl`, predict and write `submission.csv`.

**Model:** CatBoost only (5 seeds, averaged), on the same 137 columns as our main submission plus 14 identity columns used
as categories. This is our third final submission: a different architecture from the main blend (no XGBoost / LightGBM /
HistGB / logistic regression), almost as strong on cross-validation (4 fresh splits: 0.6978 vs 0.6988 for the blend).
No neural networks, no external data.

**Data usage:** `test.jsonl` is opened only in Section 6, after all models and lookup tables have been fitted, and it is
used only to compute features and predictions. The type-effectiveness table `type_chart_gen2.csv` is the one provided
with the competition data.

**Reproducibility:** all random seeds are fixed. Running the notebook top to bottom gives the same `submission.csv`.
Runtime on Kaggle CPU: about 20 minutes.

### 1. Setup and training data

In [1]:
import json, time, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore', message='X does not have valid feature names')

RANDOM_STATE = 42
SEEDS = [0, 1, 2, 3, 4]

DATA_DIR = Path('/kaggle/input/competitions/fds-pokemon-battles-prediction-2026')
if not (DATA_DIR / 'train.jsonl').exists():                       # fallback: wherever the competition data is attached
    DATA_DIR = next(Path('/kaggle/input').rglob('train.jsonl')).parent

with open(DATA_DIR / 'train.jsonl') as f:
    train_data = [json.loads(line) for line in f]
type_chart_df = pd.read_csv(DATA_DIR / 'type_chart_gen2.csv')

print(f"train: {len(train_data)} battles | data dir: {DATA_DIR}")
print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | "
      f"xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}")

train: 10000 battles | data dir: /kaggle/input/competitions/fds-pokemon-battles-prediction-2026
numpy 2.1.3 | pandas 2.3.3 | scikit-learn 1.6.1 | xgboost 3.4.1 | lightgbm 4.6.0


### 2. Lookup tables (training set only)

- **Species table:** base stats and types of every species that appears in a training battle (P1 teams and P2 leads).
  P2 Pokémon are given only by name in the timeline, so their stats come from this table; species never seen in
  training are skipped.
- **Type chart:** effectiveness multiplier for each attacking/defending type pair.
- **Moves by species:** which attack types each species used in the training battles (used to estimate how
  threatening the revealed P2 Pokémon are for the P1 team).
- **Team frequency:** how many times each exact P1 team appears in training.

In [2]:
def build_species_vocab(battles):
    vocab = {}
    for b in battles:
        for p in list(b.get('p1_team_details', [])) + [b.get('p2_lead_details') or {}]:
            name = p.get('name')
            if name and name not in vocab:
                vocab[name] = {k: p.get(k) for k in ('base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe')}
                vocab[name]['types'] = tuple(p.get('types') or [])
    return vocab

species_vocab = build_species_vocab(train_data)
print(f"species table: {len(species_vocab)} species")

TC = {(r.attacking_type, r.defending_type): r.multiplier for r in type_chart_df.itertuples()}

def type_effectiveness(att_types, def_types):
    best = 0.0
    for a in att_types:
        m = 1.0
        for d in def_types:
            m *= TC.get((a, d), 1.0)
        best = max(best, m)
    return best

def build_species_move_records(battles):
    from collections import Counter, defaultdict
    global_counts = defaultdict(Counter)
    battle_counts = defaultdict(Counter)
    for b in battles:
        bid = b.get('battle_id')
        tl = b.get('battle_timeline') or []
        for turn in tl:
            for side in ('p1', 'p2'):
                state = turn.get(f'{side}_pokemon_state') or {}
                name = state.get('name')
                mv = turn.get(f'{side}_move_details')
                if name is None or not mv:
                    continue
                if mv.get('category') == 'status' or (mv.get('base_power', 0) or 0) <= 0:
                    continue
                mv_type = mv.get('type')
                if mv_type is None:
                    continue
                global_counts[name][mv_type] += 1
                battle_counts[(name, bid)][mv_type] += 1
    return dict(global_counts), dict(battle_counts)

def _species_threat_vs_types(type_counts, def_types):
    total = sum(type_counts.values())
    if total == 0:
        return 1.0, 0
    weighted = sum(cnt * type_effectiveness([mv_type], def_types) for mv_type, cnt in type_counts.items())
    return weighted / total, total

def extract_pokedex_predictive_threat(battle, species_records, t=30):
    global_counts, battle_counts = species_records
    bid = battle.get('battle_id')
    tl = (battle.get('battle_timeline') or [])[:t]
    p1_team_types = [tuple(p.get('types') or ()) for p in battle['p1_team_details']]
    seen_p2 = {(turn.get('p2_pokemon_state') or {}).get('name') for turn in tl}
    seen_p2.discard(None)

    mean_effs, n_obs_total, n_species_with_prior = [], 0, 0
    for name in sorted(seen_p2):          # sorted: deterministic summation order
        gcounts = global_counts.get(name)
        if not gcounts:
            continue
        own = battle_counts.get((name, bid))
        loo_counts = (gcounts - own) if own else gcounts
        species_had_prior = False
        for p1_types in p1_team_types:
            eff, n_obs = _species_threat_vs_types(loo_counts, p1_types)
            if n_obs > 0:
                mean_effs.append(eff)
                n_obs_total += n_obs
                species_had_prior = True
        if species_had_prior:
            n_species_with_prior += 1

    return {
        'p2_pred_threat_mean_eff_vs_p1_bench': float(np.mean(mean_effs)) if mean_effs else 1.0,
        'p2_pred_threat_max_eff_vs_p1_bench': float(np.max(mean_effs)) if mean_effs else 1.0,
        'p2_pred_threat_n_species_with_prior': n_species_with_prior,
        'p2_pred_threat_total_prior_obs': n_obs_total,
    }

pokedex_predictive_threat_cols = ['p2_pred_threat_mean_eff_vs_p1_bench', 'p2_pred_threat_max_eff_vs_p1_bench',
                                   'p2_pred_threat_n_species_with_prior', 'p2_pred_threat_total_prior_obs']

species_records = build_species_move_records(train_data)

def team_signature(battle):
    return "|".join(sorted(p["name"] for p in battle["p1_team_details"]))

team_sig_train = pd.Series([team_signature(b) for b in train_data], name="team_signature")
team_sig_counts = team_sig_train.value_counts().to_dict()

def extract_team_frequency(battle, counts_lookup):
    sig = team_signature(battle)
    return {'team_freq_count': counts_lookup.get(sig, 0)}

print(f"team frequency table: {len(team_sig_counts)} distinct P1 teams")

species table: 239 species
team frequency table: 4167 distinct P1 teams


### 3. Feature engineering

Every battle becomes one row of 134 numbers (122 below + 6 duels in 3.5 + 6 P2-team features in 3.7), computed only on the 30 observed turns. Most features are P1 − P2
differences.

| Block | # | What it describes |
|---|---:|---|
| Baseline | 57 | team stats, HP and status over time, KOs and who scored first, team-HP trend, revealed P2 profile, expected threat of P2's attacks against P1's team, P1 team frequency |
| Material and matchup (A-I) | 36 | Pokémon alive at turn 30, HP weighted by status and by species strength, the two active Pokémon (speed, type advantage, boosts), Spikes on the field, recent HP momentum, Leftovers |
| Volatile state (K) | 14 | substitute, Leech Seed, Perish Song, confusion, trapping, frozen Pokémon, Reflect/Light Screen |
| Sustain (DS) | 5 | walls with Leftovers still alive, alive Pokémon that used Rest or recovery moves |
| Pairwise duels | 6 | Gen 2 damage formula: which side wins the one-on-one matchups between alive Pokémon |
| Setup dynamics | 10 | how boosts evolved: stages gained/lost, setup moves (total and in turns 21-30), HP paid while setting up, boosts lost by switching, attacks made while boosted, how dangerous the active Pokémon is at turn 30 |

#### 3.1 Baseline features

In [3]:
def extract_momentum_ko(battle, t=30):
    tl = (battle.get('battle_timeline') or [])[:t]
    n = len(tl)
    if n < t:
        return {'mom_ko_adv_first15': 0, 'mom_ko_first_side': 0}
    ko = {}
    for side in ('p1', 'p2'):
        observed_faints = set()
        kos = np.zeros(n)
        for i, turn in enumerate(tl):
            state = turn.get(f'{side}_pokemon_state') or {}
            name = state.get('name')
            hp = state.get('hp_pct')
            if name is not None and ((hp is not None and hp <= 0) or state.get('status') == 'fnt'):
                observed_faints.add(name)
            kos[i] = len(observed_faints)
        ko[side] = kos
    K = ko['p2'] - ko['p1']
    first_ko_side = 0
    for i in range(n):
        d1 = ko['p1'][i] - (ko['p1'][i - 1] if i else 0)
        d2 = ko['p2'][i] - (ko['p2'][i - 1] if i else 0)
        if d1 > 0 or d2 > 0:
            first_ko_side = int(np.sign(d2 - d1))
            break
    return {'mom_ko_adv_first15': float(K[14]), 'mom_ko_first_side': first_ko_side}


DROP_FROM_BASELINE = {'n_turns', 'p1_fainted', 'diff_fainted'}

def extract_features(battle):
    f = {}
    p1_team = battle.get('p1_team_details', [])
    if p1_team:
        f['p1_mean_hp'] = np.mean([p.get('base_hp', 0) for p in p1_team])
        f['p1_mean_spe'] = np.mean([p.get('base_spe', 0) for p in p1_team])
        f['p1_mean_atk'] = np.mean([p.get('base_atk', 0) for p in p1_team])
        f['p1_mean_def'] = np.mean([p.get('base_def', 0) for p in p1_team])
    p2_lead = battle.get('p2_lead_details')
    if p2_lead:
        f['p2_lead_hp'] = p2_lead.get('base_hp', 0)
        f['p2_lead_spe'] = p2_lead.get('base_spe', 0)

    tl = battle.get('battle_timeline', []) or []
    f['n_turns'] = len(tl)
    side_stats = {}
    for side in ['p1', 'p2']:
        states = [t.get(f'{side}_pokemon_state') for t in tl]
        states = [s for s in states if s]
        if not states:
            continue
        hp = np.array([s['hp_pct'] for s in states])
        names = [s['name'] for s in states]
        last_hp = {}
        for s in states:
            last_hp[s['name']] = s['hp_pct']
        n_seen = len(last_hp)

        side_f = {}
        side_f['team_hp_sum'] = sum(last_hp.values()) + max(0, 6 - n_seen) * 1.0
        side_f['fainted'] = sum(v == 0 for v in last_hp.values())
        side_f['n_seen'] = n_seen
        side_f['switches'] = sum(a != b for a, b in zip(names, names[1:]))
        side_f['hp_last'] = hp[-1]
        side_f['hp_mean'] = hp.mean()
        side_f['hp_min'] = hp.min()
        statused = [s['status'] != 'nostatus' for s in states]
        side_f['status_share'] = np.mean(statused)
        side_f['n_statused'] = len({s['name'] for s in states if s['status'] != 'nostatus'})
        bsum = [sum(s['boosts'].values()) for s in states]
        side_f['boost_last'] = bsum[-1]
        side_f['boost_mean'] = np.mean(bsum)
        moves = [t.get(f'{side}_move_details') for t in tl]
        used = [m for m in moves if m]
        side_f['no_move_share'] = 1 - len(used) / len(tl)
        side_f['status_move_share'] = np.mean([m['category'] == 'status' for m in used]) if used else 0
        side_f['mean_bp'] = np.mean([m['base_power'] for m in used]) if used else 0

        side_stats[side] = side_f
        for k, v in side_f.items():
            f[f'{side}_{k}'] = v

    if 'p1' in side_stats and 'p2' in side_stats:
        for k in ['team_hp_sum', 'fainted', 'hp_last', 'hp_mean', 'status_share',
                  'n_statused', 'boost_mean', 'switches']:
            f[f'diff_{k}'] = side_stats['p1'][k] - side_stats['p2'][k]

    f.update(extract_momentum_ko(battle))
    for k in DROP_FROM_BASELINE:
        f.pop(k, None)
    f['battle_id'] = battle.get('battle_id')
    if 'player_won' in battle:
        f['player_won'] = int(battle['player_won'])
    return f


def extract_team_hp_trend(battle, t=30):
    tl = (battle.get('battle_timeline') or [])[:t]
    n = len(tl)
    series = {}
    for side in ('p1', 'p2'):
        last_hp = {}
        vals = np.zeros(n)
        for i, turn in enumerate(tl):
            state = turn.get(f'{side}_pokemon_state') or {}
            name, hp = state.get('name'), state.get('hp_pct')
            if name is not None and hp is not None:
                last_hp[name] = hp
            n_seen = len(last_hp)
            vals[i] = sum(last_hp.values()) + max(0, 6 - n_seen) * 1.0
        series[side] = vals

    diff = series['p1'] - series['p2']
    out = {}
    if n >= 2:
        x = np.arange(1, n + 1)
        slope_p1 = np.polyfit(x, series['p1'], 1)[0]
        slope_p2 = np.polyfit(x, series['p2'], 1)[0]
    else:
        slope_p1 = slope_p2 = 0.0
    out['diff_team_hp_slope'] = float(slope_p1 - slope_p2)
    for snap_t in (10, 20):
        idx = min(snap_t - 1, n - 1)
        out[f'diff_team_hp_t{snap_t}'] = float(diff[idx]) if n else 0.0
    out['diff_team_hp_min'] = float(diff.min()) if n else 0.0
    out['diff_team_hp_std'] = float(diff.std()) if n else 0.0
    half = n // 2
    out['diff_team_hp_momentum'] = float(diff[half:].mean() - diff[:half].mean()) if half else 0.0
    return out

team_hp_trend_cols = ['diff_team_hp_slope', 'diff_team_hp_t10', 'diff_team_hp_t20',
                       'diff_team_hp_min', 'diff_team_hp_std', 'diff_team_hp_momentum']


def extract_pokedex_revealed_team_profile(battle, vocab, t=30):
    tl = (battle.get('battle_timeline') or [])[:t]
    seen_names, seen_recs = set(), []
    for turn in tl:
        name = (turn.get('p2_pokemon_state') or {}).get('name')
        if name and name not in seen_names:
            rec = vocab.get(name)
            if rec:
                seen_names.add(name)
                seen_recs.append(rec)
    if not seen_recs:
        return {'p2_profile_mean_spe': 0.0, 'p2_profile_mean_atk_or_spa': 0.0,
                'p2_profile_frac_fast_attacker': 0.0, 'p2_profile_n_resolved': 0}
    spe = [r['base_spe'] for r in seen_recs]
    atk_or_spa = [max(r['base_atk'], r['base_spa']) for r in seen_recs]
    fast_attacker = [int(r['base_spe'] >= 90 and max(r['base_atk'], r['base_spa']) >= 90) for r in seen_recs]
    return {
        'p2_profile_mean_spe': float(np.mean(spe)),
        'p2_profile_mean_atk_or_spa': float(np.mean(atk_or_spa)),
        'p2_profile_frac_fast_attacker': float(np.mean(fast_attacker)),
        'p2_profile_n_resolved': len(seen_recs),
    }

pokedex_team_profile_cols = ['p2_profile_mean_spe', 'p2_profile_mean_atk_or_spa',
                              'p2_profile_frac_fast_attacker', 'p2_profile_n_resolved']

#### 3.2 Material, matchup, volatile state and sustain

In [4]:
def bst(rec): return sum(rec[k] for k in ('base_hp','base_atk','base_def','base_spa','base_spd','base_spe'))
STATUS_PEN = {'nostatus':1.0,'par':0.75,'tox':0.75,'psn':0.85,'brn':0.8,'slp':0.5,'frz':0.1}
def eff1(att, def_types):
    m = 1.0
    for d in def_types: m *= TC.get((att, d), 1.0)
    return m

_p2c = Counter(t['p2_pokemon_state']['name'] for b in train_data for t in b['battle_timeline'])
P2_AVG_BST = np.average([bst(species_vocab[n]) for n in _p2c if n in species_vocab],
                        weights=[_p2c[n] for n in _p2c if n in species_vocab])

def final_states(b, side):
    last = {}
    for t in b['battle_timeline']:
        s = t.get(f'{side}_pokemon_state') or {}
        if s.get('name'): last[s['name']] = s
    return last
def is_alive(s): return (s.get('hp_pct') or 0) > 0 and s.get('status') != 'fnt'

def extract_domain(b):
    f = {}; tl = b['battle_timeline']
    team = {p['name']: p for p in b['p1_team_details']}
    F = {'p1': final_states(b, 'p1'), 'p2': final_states(b, 'p2')}
    # A: material at turn 30
    for side in ('p1','p2'):
        fainted = sum(1 for s in F[side].values() if not is_alive(s))
        f[f'A_{side}_fainted'] = fainted; f[f'A_{side}_alive'] = 6 - fainted
    f['A_diff_alive'] = f['A_p1_alive'] - f['A_p2_alive']
    # B: final status burden + effective HP
    for side in ('p1','p2'):
        eh, bad, sf, par = 0.0, 0, 0, 0
        for s in F[side].values():
            if not is_alive(s): continue
            st = s.get('status') or 'nostatus'
            eh += s['hp_pct'] * STATUS_PEN.get(st, 1.0); bad += st != 'nostatus'; sf += st in ('slp','frz'); par += st == 'par'
        eh += (6 - len(F[side]))
        f[f'B_{side}_eff_hp'], f[f'B_{side}_alive_statused'], f[f'B_{side}_alive_slp_frz'], f[f'B_{side}_alive_par'] = eh, bad, sf, par
    for k in ('eff_hp', 'alive_statused', 'alive_slp_frz'):
        f[f'B_diff_{k}'] = f[f'B_p1_{k}'] - f[f'B_p2_{k}']
    # C: quality-weighted remaining HP
    q1 = sum((F['p1'][n]['hp_pct'] if n in F['p1'] else 1.0) * bst(p) / 600 for n, p in team.items() if n not in F['p1'] or is_alive(F['p1'][n]))
    q2 = sum(s['hp_pct'] * (bst(species_vocab[n]) if n in species_vocab else P2_AVG_BST) / 600 for n, s in F['p2'].items() if is_alive(s))
    q2 += (6 - len(F['p2'])) * P2_AVG_BST / 600
    f['C_p1_quality_hp'], f['C_p2_quality_hp'], f['C_diff_quality_hp'] = q1, q2, q1 - q2
    # D: active matchup at turn 30
    a1, a2 = tl[-1]['p1_pokemon_state'], tl[-1]['p2_pokemon_state']
    r1, r2 = species_vocab.get(a1['name'], {}), species_vocab.get(a2['name'], {})
    def eff_spe(a, r):
        bo = a['boosts'].get('spe', 0)
        return r.get('base_spe', 80) * (0.25 if a['status'] == 'par' else 1) * (1 + 0.5 * max(0, bo)) / (1 + 0.5 * max(0, -bo))
    f['D_active_p1_faster'] = float(np.sign(eff_spe(a1, r1) - eff_spe(a2, r2)))
    f['D_active_hp_diff'] = (a1['hp_pct'] or 0) - (a2['hp_pct'] or 0)
    t1, t2 = r1.get('types', ()), r2.get('types', ())
    f['D_active_p1_stab_eff'] = max([eff1(t, t2) for t in t1 if t != 'notype'] or [1.0])
    f['D_active_p2_stab_eff'] = max([eff1(t, t1) for t in t2 if t != 'notype'] or [1.0])
    f['D_active_diff_eff'] = f['D_active_p1_stab_eff'] - f['D_active_p2_stab_eff']
    for k in ('atk','def','spa','spd','spe'):
        f[f'D_active_diff_boost_{k}'] = a1['boosts'].get(k, 0) - a2['boosts'].get(k, 0)
    # E: spikes on the field (set and not spun away)
    on = {'p1': False, 'p2': False}
    for t in tl:
        for side, opp in (('p1','p2'), ('p2','p1')):
            mv = (t.get(f'{side}_move_details') or {}).get('name')
            if mv == 'spikes': on[opp] = True
            if mv == 'rapidspin': on[side] = False
    f['E_spikes_on_p1'], f['E_spikes_on_p2'] = int(on['p1']), int(on['p2'])
    f['E_diff_spikes'] = f['E_spikes_on_p2'] - f['E_spikes_on_p1']
    # I: leftovers on alive P1 mons
    f['I_p1_alive_leftovers'] = sum(1 for n, p in team.items() if p.get('item') == 'leftovers' and (n not in F['p1'] or is_alive(F['p1'][n])))
    # H: short-term team-HP momentum
    def series(side):
        last, out = {}, []
        for t in tl:
            s = t[f'{side}_pokemon_state']; last[s['name']] = s['hp_pct'] or 0.0
            out.append(sum(last.values()) + (6 - len(last)))
        return np.array(out)
    d = series('p1') - series('p2')
    f['H_diff_hp_delta3'], f['H_diff_hp_delta5'], f['H_diff_hp_last5_mean'] = d[-1] - d[-4], d[-1] - d[-6], d[-5:].mean()
    # K: volatile state of the actives at turn 30, frozen mons, screens in the last 5 turns
    for e in ('leechseed', 'substitute', 'perish', 'confusion', 'trapped'):
        f[f'K_active_p1_{e}'] = int(e in (a1.get('effects') or []))
        f[f'K_active_p2_{e}'] = int(e in (a2.get('effects') or []))
    f['K_p1_alive_frz'] = sum(1 for n in team if n in F['p1'] and is_alive(F['p1'][n]) and F['p1'][n]['status'] == 'frz')
    f['K_p2_alive_frz'] = sum(1 for s in F['p2'].values() if is_alive(s) and s['status'] == 'frz')
    for side in ('p1','p2'):
        f[f'K_{side}_screen_last5'] = int(any((t.get(f'{side}_move_details') or {}).get('name') in ('reflect', 'lightscreen') for t in tl[-5:]))
    return f


SUSTAIN_MOVES = {'rest', 'sleeptalk', 'recover', 'softboiled', 'milkdrink'}

def extract_sustain(b):
    # DS: defensive sustain, from the user's loss analysis (hypothesis 4)
    tl = b['battle_timeline']; team = {p['name']: p for p in b['p1_team_details']}
    F = {'p1': final_states(b, 'p1'), 'p2': final_states(b, 'p2')}
    used = {'p1': defaultdict(set), 'p2': defaultdict(set)}
    for t in tl:
        for side in ('p1', 'p2'):
            mv = t.get(f'{side}_move_details')
            if mv and mv.get('name'): used[side][t[f'{side}_pokemon_state']['name']].add(mv['name'])
    f = {}
    f['DS_p1_wall_left_hp'] = sum((F['p1'][n]['hp_pct'] if n in F['p1'] else 1.0) for n, p in team.items()
                                  if max(p['base_def'], p['base_spd']) >= 120 and p.get('item') == 'leftovers'
                                  and (n not in F['p1'] or is_alive(F['p1'][n])))
    for side in ('p1', 'p2'):
        f[f'DS_{side}_sustain_alive'] = sum(1 for n, mv in used[side].items() if n in F[side] and is_alive(F[side][n]) and mv & SUSTAIN_MOVES)
    f['DS_diff_sustain_alive'] = f['DS_p1_sustain_alive'] - f['DS_p2_sustain_alive']
    a1, a2 = tl[-1]['p1_pokemon_state'], tl[-1]['p2_pokemon_state']
    f['DS_active_diff_defboost'] = sum(a1['boosts'].get(k, 0) for k in ('def', 'spd')) - sum(a2['boosts'].get(k, 0) for k in ('def', 'spd'))
    return f

#### 3.3 Setup dynamics

Computed turn by turn on the active Pokémon of each side. Boosts disappear when a Pokémon leaves the field, so they are
never attributed to the bench.

In [5]:
BOOST_STATS = ['atk', 'def', 'spa', 'spd', 'spe']
SETUP_MOVES = {'curse', 'swordsdance', 'bellydrum', 'growth', 'amnesia', 'meditate', 'sharpen',
               'defensecurl', 'harden', 'acidarmor', 'agility'}
SETUP_COLS = ['rise_events', 'drop_events', 'setup_move_count', 'setup_late', 'setup_switches', 'setup_hit_count',
              'setup_cost_hp', 'last_boost_gap', 'active_setup_risk', 'boost_density_episode']

def side_setup(battle, side, t=30):
    """Dinamica dei boost di un lato, solo dal Pokemon in campo turno per turno (niente boost attribuiti alla panchina)."""
    f = dict.fromkeys(SETUP_COLS, 0.0); f['last_boost_gap'] = 30.0
    prev = None; ep_turns = ep_boosted = 0; hp_deltas = []; last_setup_turn = 0; last = None
    for j, turn in enumerate((battle.get('battle_timeline') or [])[:t]):
        st = turn.get(f'{side}_pokemon_state') or {}; mv = turn.get(f'{side}_move_details') or {}
        name = st.get('name')
        if not name:
            continue
        b = st.get('boosts') or {}
        v = np.array([int(b.get(k, 0)) for k in BOOST_STATS], dtype=float)
        pos = float(np.maximum(v, 0).sum()); hp = max(0.0, float(st.get('hp_pct', 1.0) or 0.0))
        move = mv.get('name') or ''; bp = float(mv.get('base_power') or 0)
        if prev is not None and prev['name'] != name:            # cambio: l'episodio riparte
            f['setup_switches'] += prev['pos'] > 0                 # boost persi uscendo dal campo
            ep_turns = ep_boosted = 0
        ep_turns += 1; ep_boosted += pos > 0
        if move in SETUP_MOVES:
            f['setup_move_count'] += 1; f['setup_late'] += j >= 20; last_setup_turn = j + 1
            if prev is not None and prev['name'] == name:
                hp_deltas.append(hp - prev['hp'])                  # HP persi nel turno di setup
        if pos > 0 and bp > 0:
            f['setup_hit_count'] += 1                              # attacchi sferrati da potenziato
        if prev is not None and prev['name'] == name:
            dv = v - prev['vec']
            f['rise_events'] += float(np.maximum(dv, 0).sum()); f['drop_events'] += float(np.maximum(-dv, 0).sum())
        prev = {'name': name, 'vec': v, 'pos': pos, 'hp': hp}; last = (j + 1, pos, hp, move)
    if last is None:
        return f
    f['setup_cost_hp'] = float(np.mean(hp_deltas)) if hp_deltas else 0.0
    f['last_boost_gap'] = float(last[0] - last_setup_turn) if last_setup_turn else 30.0
    f['active_setup_risk'] = last[1] * last[2] * (1 + (last[3] in SETUP_MOVES))
    f['boost_density_episode'] = ep_boosted / max(1, ep_turns)
    return f

def extract_setup(battle):
    a, b = side_setup(battle, 'p1'), side_setup(battle, 'p2')
    return {f'diff_{k}': a[k] - b[k] for k in SETUP_COLS}

#### 3.4 Training feature matrix

The "expected threat" feature uses the moves each species used in the *other* training battles (the current battle is
removed, so a training row never sees its own moves through this table).

In [6]:
def build_feature_row(battle, records):
    f = extract_features(battle)
    f.update(extract_team_hp_trend(battle))
    f.update(extract_pokedex_revealed_team_profile(battle, species_vocab))
    f.update(extract_pokedex_predictive_threat(battle, records))
    f.update(extract_team_frequency(battle, team_sig_counts))
    f.update(extract_domain(battle))
    f.update(extract_sustain(battle))
    f.update(extract_setup(battle))
    return f

train_df = pd.DataFrame([build_feature_row(b, species_records) for b in train_data]).fillna(0)
feature_cols = [c for c in train_df.columns if c not in ('battle_id', 'player_won')]
assert len(feature_cols) == 122, len(feature_cols)
X_train = train_df[feature_cols].values.astype(float)
y = train_df['player_won'].values.astype(int)
print(f"X_train: {X_train.shape} | P1 win rate: {y.mean():.3f}")

X_train: (10000, 122) | P1 win rate: 0.500


#### 3.5 Pairwise duels (6 features)

For every alive P1 Pokémon and every alive revealed P2 Pokémon: who would knock the other out first, using the Gen 2
damage formula (level 100, base stats, types, STAB, type effectiveness), calibrated on the damage observed in training
battles (×0.85; no labels involved). Moves: those used in this battle, otherwise the species' 3 most used attacks in
training. Turns to KO from current HP, +2 if asleep or frozen, ties broken by speed (paralysis ×0.25).

| Feature | Meaning |
|---|---|
| `DU_pair_win_share` | share of (P1, P2) pairs won by P1 |
| `DU_p2_covered` / `DU_p2_uncovered` | alive revealed P2 Pokémon that P1 can / cannot beat with at least one Pokémon |
| `DU_p1_covered` / `DU_p1_safe` | alive P1 Pokémon that P2 can / cannot beat with at least one Pokémon |
| `DU_net` | `DU_p1_safe − DU_p2_uncovered` |

In [7]:
# Gen 2 damage estimate (level 100, max DVs/stat exp), calibrated on observed training damage (x0.85, no labels used)
FIXED_DMG = {'seismictoss', 'nightshade'}                       # 100 HP of damage at level 100
NO_ESTIMATE = {'counter', 'mirrorcoat', 'superfang', 'dragonrage', 'sonicboom', 'explosion', 'selfdestruct', 'hiddenpower'}
DMG_CAL = 0.85

def gen2_stat(base): return 2 * base + 98
def gen2_hp(base): return 2 * base + 203
def stage_mult(s): return (2 + s) / 2 if s >= 0 else 2 / (2 - s)
def is_attack(m): return bool(m) and (m.get('base_power') or m['name'] in FIXED_DMG) and m['name'] not in NO_ESTIMATE

_move_counts, _move_info = defaultdict(Counter), {}
for b in train_data:                                            # typical attacks of each species (training set only)
    for t in b['battle_timeline']:
        for s in ('p1', 'p2'):
            m = t.get(f'{s}_move_details')
            if is_attack(m):
                _move_counts[t[f'{s}_pokemon_state']['name']][m['name']] += 1; _move_info[m['name']] = m
TYPICAL_ATTACKS = {sp: [_move_info[m] for m, _ in c.most_common(3)] for sp, c in _move_counts.items()}

def damage_frac(att, dfn, move, att_boosts=None, def_boosts=None, burned=False):
    A, D = species_vocab.get(att), species_vocab.get(dfn)
    if not A or not D:
        return 0.0
    if move['name'] in FIXED_DMG:
        return 100 / gen2_hp(D['base_hp'])
    ab, db = att_boosts or {}, def_boosts or {}
    phys = move['category'] == 'physical'
    a = gen2_stat(A['base_atk' if phys else 'base_spa']) * stage_mult(ab.get('atk' if phys else 'spa', 0)) * (0.5 if burned and phys else 1)
    d = gen2_stat(D['base_def' if phys else 'base_spd']) * stage_mult(db.get('def' if phys else 'spd', 0))
    eff = float(np.prod([TC.get((move['type'], t), 1.0) for t in D['types']]))
    stab = 1.5 if move['type'] in A['types'] else 1.0
    return DMG_CAL * ((42 * move['base_power'] * a / d) / 50 + 2) * stab * eff * 0.925 / gen2_hp(D['base_hp'])

def best_damage(att, dfn, known_moves):
    moves = known_moves or TYPICAL_ATTACKS.get(att, [])      # moves seen in this battle, else the species' typical ones
    return max([damage_frac(att, dfn, m) for m in moves] or [0.0])

def turns_to_ko(hp, dmg): return min(10.0, float(np.ceil(hp / dmg))) if dmg > 0 else 10.0

def extract_duels(battle):
    # every alive P1 Pokemon vs every alive revealed P2 Pokemon: who knocks the other out first (HP, speed, sleep/freeze)
    tl = battle['battle_timeline']
    moves = {'p1': defaultdict(dict), 'p2': defaultdict(dict)}; last = {'p1': {}, 'p2': {}}
    for t in tl:
        for s in ('p1', 'p2'):
            st = t[f'{s}_pokemon_state']; last[s][st['name']] = st; m = t.get(f'{s}_move_details')
            if is_attack(m):
                moves[s][st['name']][m['name']] = m
    def state(side, n):
        st = last[side].get(n)
        hp, status = (st['hp_pct'], st['status']) if st else (1.0, 'nostatus')
        spe = species_vocab.get(n, {}).get('base_spe', 80) * (0.25 if status == 'par' else 1)
        return hp, spe, status in ('slp', 'frz')
    P1 = [p['name'] for p in battle['p1_team_details'] if p['name'] not in last['p1'] or is_alive(last['p1'][p['name']])]
    P2 = [n for n, s in last['p2'].items() if is_alive(s)]
    W = np.zeros((len(P1), len(P2)))
    for i, a in enumerate(P1):
        h1, s1, z1 = state('p1', a)
        for j, c in enumerate(P2):
            h2, s2, z2 = state('p2', c)
            t1 = turns_to_ko(h2, best_damage(a, c, list(moves['p1'][a].values()))) + 2 * z1
            t2 = turns_to_ko(h1, best_damage(c, a, list(moves['p2'][c].values()))) + 2 * z2
            W[i, j] = 1.0 if t1 < t2 or (t1 == t2 and s1 > s2) else 0.0
    f = {'DU_pair_win_share': W.mean() if W.size else 0.5,
         'DU_p2_covered': float((W.max(0) > 0).sum()) if W.size else 0.0,
         'DU_p1_covered': float(((1 - W).max(1) > 0).sum()) if W.size else 0.0}
    f['DU_p2_uncovered'] = len(P2) - f['DU_p2_covered']
    f['DU_p1_safe'] = len(P1) - f['DU_p1_covered']
    f['DU_net'] = f['DU_p1_safe'] - f['DU_p2_uncovered']
    return f

duel_train = pd.DataFrame([extract_duels(b) for b in train_data])
print(f"duels: {duel_train.shape}")

duels: (10000, 6)


#### 3.6 Battle tokens (1 rating column)

Each battle becomes a bag of signed tokens, P1 positive and P2 negative:
- **material by identity** at turn 30: each species weighted by its remaining HP (0 if fainted, 1 if never seen for P1);
- **status by species** at turn 30 (e.g. Snorlax asleep, Zapdos paralysed);
- **moves used** in the 30 turns (bag of moves, /30).

A regularised logistic regression (C = 0.3) on these ~1,100 tokens gives a "token margin" that becomes one feature. It
uses the labels, so it is computed exactly like the strength ratings: out-of-fold inside every training set, fitted on
the whole training set for the rows to predict. The token list comes from the training set only.

In [8]:
def battle_tokens(b):
    T = Counter()
    L1, L2 = final_states(b, 'p1'), final_states(b, 'p2')
    for p in b['p1_team_details']:
        s = L1.get(p['name'])
        T[('mat', p['name'])] += 1.0 if s is None else (0.0 if s['status'] == 'fnt' else s['hp_pct'])
        if s is not None and s['status'] not in ('nostatus', 'fnt'):
            T[('st', p['name'], s['status'])] += 1
    for n, s in L2.items():
        T[('mat', n)] -= 0.0 if s['status'] == 'fnt' else s['hp_pct']
        if s['status'] not in ('nostatus', 'fnt'):
            T[('st', n, s['status'])] -= 1
    for t in b['battle_timeline']:
        for side, sign in (('p1', 1), ('p2', -1)):
            m = t.get(f'{side}_move_details')
            if m:
                T[('mv', m['name'])] += sign / 30.0
    return T

train_tokens = [battle_tokens(b) for b in train_data]
TOKEN_VOCAB = {}
for T in train_tokens:
    for k, v in T.items():
        if v != 0:
            TOKEN_VOCAB.setdefault(k, len(TOKEN_VOCAB))

def token_matrix(token_list):
    r, c, v = [], [], []
    for i, T in enumerate(token_list):
        for k, val in T.items():
            if val != 0 and k in TOKEN_VOCAB:
                r.append(i); c.append(TOKEN_VOCAB[k]); v.append(val)
    return sparse.csr_matrix((v, (r, c)), shape=(len(token_list), len(TOKEN_VOCAB)))

def tok_margin(M_fit, y_fit, M_apply):
    return LogisticRegression(C=0.3, max_iter=4000).fit(M_fit, y_fit).decision_function(M_apply)

def tok_inner_oof(M, y_fit):
    out = np.zeros(len(y_fit))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(len(y_fit)), y_fit):
        out[b] = tok_margin(M[a], y_fit[a], M[b])
    return out

TM_train = token_matrix(train_tokens)
print(f"token matrix: {TM_train.shape}")

token matrix: (10000, 1096)


#### 3.7 P2's team when identified (6 features + 8 identity columns)

When the Pokémon revealed by P2 match exactly one P1 team seen in training (≈48% of battles), P2's whole team and its
usual items are known. Features: identified flag, P2 Leftovers still in play, P2 walls with Leftovers (HP left),
strength and number of the Pokémon P2 has not shown yet, P1 − P2 Leftovers in play. `NaN` when the team is not
identified. Teams and items come from the training set only.
The 8 identity columns (active Pokémon at turn 30, leads, P1 team, identified P2 team, status of the actives) are given
to CatBoost as categorical features.

In [9]:
# Known teams (training set only) and the items they usually hold
team_items = defaultdict(Counter)
for b in train_data:
    T = frozenset(p['name'] for p in b['p1_team_details'])
    team_items[T][tuple(sorted((p['name'], p.get('item')) for p in b['p1_team_details']))] += 1
known_teams = list(team_items)
teams_with_species = defaultdict(set)
for i, T in enumerate(known_teams):
    for s_ in T:
        teams_with_species[s_].add(i)

def p2_revealed_names(battle):
    s_ = {t['p2_pokemon_state']['name'] for t in battle['battle_timeline']}
    s_.add(battle['p2_lead_details']['name'])
    return s_

def identify_p2_team(battle):
    """P2's team if the Pokémon it revealed match exactly one known team, else None."""
    R = p2_revealed_names(battle)
    if not all(s_ in teams_with_species for s_ in R):
        return None
    cand = set.intersection(*(teams_with_species[s_] for s_ in R))
    return known_teams[next(iter(cand))] if len(cand) == 1 else None

P2_TEAM_COLS = ['P2T_identified', 'P2T_alive_leftovers', 'P2T_wall_left_hp', 'P2T_unseen_bst_mean',
                'P2T_n_unseen', 'P2T_diff_alive_leftovers']

def extract_p2_identified(b):
    nan = float('nan')
    T = identify_p2_team(b)
    if T is None:
        return dict(zip(P2_TEAM_COLS, [0, nan, nan, nan, nan, nan]))
    team = {p['name']: p for p in b['p1_team_details']}
    F1, F2 = final_states(b, 'p1'), final_states(b, 'p2')
    p1_left = sum(1 for n, p in team.items() if p.get('item') == 'leftovers' and (n not in F1 or is_alive(F1[n])))
    items = dict(team_items[T].most_common(1)[0][0])
    alive = lambda n: (n not in F2) or is_alive(F2[n])
    lefties = sum(1 for n in T if items.get(n) == 'leftovers' and alive(n))
    walls = sum((F2[n]['hp_pct'] if n in F2 else 1.0) for n in T if n in species_vocab and alive(n)
                and max(species_vocab[n]['base_def'], species_vocab[n]['base_spd']) >= 120 and items.get(n) == 'leftovers')
    R = p2_revealed_names(b)
    unseen = [n for n in T if n not in R and n in species_vocab]
    unseen_bst = float(np.mean([bst(species_vocab[n]) for n in unseen])) if unseen else nan
    return dict(zip(P2_TEAM_COLS, [1, lefties, walls, unseen_bst, len(unseen), p1_left - lefties]))


CAT_COLS = ['p1_active_30', 'p2_active_30', 'p1_lead', 'p2_lead', 'p1_team', 'p2_team', 'p1_status_30', 'p2_status_30']

def categorical_row(b):
    """Identity columns for CatBoost (inputs only)."""
    tl, T = b['battle_timeline'], identify_p2_team(b)
    return {'p1_active_30': tl[-1]['p1_pokemon_state']['name'], 'p2_active_30': tl[-1]['p2_pokemon_state']['name'],
            'p1_lead': tl[0]['p1_pokemon_state']['name'], 'p2_lead': b['p2_lead_details']['name'],
            'p1_team': '|'.join(sorted(p['name'] for p in b['p1_team_details'])),
            'p2_team': '|'.join(sorted(T)) if T is not None else 'unknown',
            'p1_status_30': tl[-1]['p1_pokemon_state']['status'], 'p2_status_30': tl[-1]['p2_pokemon_state']['status']}

X_p2 = pd.DataFrame([extract_p2_identified(b) for b in train_data])[P2_TEAM_COLS].values.astype(float)
C_train = pd.DataFrame([categorical_row(b) for b in train_data])[CAT_COLS]
print(f"known teams: {len(known_teams)} | P2 team identified in {X_p2[:, 0].mean():.1%} of battles")

known teams: 4167 | P2 team identified in 47.5% of battles


#### 3.8 Extra identity columns for CatBoost

Six more categorical columns, all computed from the battle itself (no labels):

| Column | Meaning |
|---|---|
| `pair_active_30` | P1 active vs P2 active at turn 30, as one category (the matchup on the field) |
| `p1_alive_set` | P1 Pokémon still alive at turn 30 (including those never sent out) |
| `p2_seen_set` | P2 Pokémon revealed so far |
| `p2_alive_seen` | revealed P2 Pokémon still alive |
| `p1_last_move` / `p2_last_move` | last move used by each side (`none` if no move) |

In [10]:
def extra_categorical_row(b):
    tl = b['battle_timeline']
    F1, F2 = final_states(b, 'p1'), final_states(b, 'p2')
    def last_move(side):
        for t in reversed(tl):
            m = t.get(f'{side}_move_details')
            if m: return m['name']
        return 'none'
    alive1 = sorted(p['name'] for p in b['p1_team_details'] if p['name'] not in F1 or is_alive(F1[p['name']]))
    return {'pair_active_30': tl[-1]['p1_pokemon_state']['name'] + '>' + tl[-1]['p2_pokemon_state']['name'],
            'p1_alive_set': '|'.join(alive1),
            'p2_seen_set': '|'.join(sorted(p2_revealed_names(b))),
            'p2_alive_seen': '|'.join(sorted(n for n, s_ in F2.items() if is_alive(s_))),
            'p1_last_move': last_move('p1'), 'p2_last_move': last_move('p2')}

EXTRA_COLS = ['pair_active_30', 'p1_alive_set', 'p2_seen_set', 'p2_alive_seen', 'p1_last_move', 'p2_last_move']
C_extra = pd.DataFrame([extra_categorical_row(b) for b in train_data])[EXTRA_COLS]
C_plus = pd.concat([C_train, C_extra], axis=1)
print("categorical columns for CatBoost:", len(C_plus.columns), "| distinct values:", {c: C_plus[c].nunique() for c in C_plus.columns})

categorical columns for CatBoost: 14 | distinct values: {'p1_active_30': 173, 'p2_active_30': 172, 'p1_lead': 177, 'p2_lead': 161, 'p1_team': 4167, 'p2_team': 1327, 'p1_status_30': 8, 'p2_status_30': 8, 'pair_active_30': 2242, 'p1_alive_set': 5669, 'p2_seen_set': 5202, 'p2_alive_seen': 5851, 'p1_last_move': 155, 'p2_last_move': 155}


### 4. Strength ratings for teams and species

P1 and P2 draw their teams from the same pool, and in about half of the battles the Pokémon revealed by P2 identify
its team among the P1 teams seen in training. We fit a regularised logistic regression on two sparse matrices:

- **teams:** +1 on P1's team, −1/k on each of the k training teams compatible with P2's revealed Pokémon (only if 1 ≤ k ≤ 5);
- **species:** +1 for each P1 species, −1 for each revealed P2 species.

Its output (the "strength margin" of P1 over P2) becomes 2 extra features. The team and species lists come from the
training set only.

These ratings use the labels, so they must never see the row they are computed for. Inside any training set, each row
gets its rating from a model fitted on the other 4/5 of that set (inner 5-fold); rows to predict get it from a model
fitted on the whole training set.

In [11]:
pool = Counter(frozenset(p['name'] for p in b['p1_team_details']) for b in train_data)
team_idx = {T: i for i, T in enumerate(pool)}
by_species = defaultdict(set)
for T, i in team_idx.items():
    for s in T:
        by_species[s].add(i)
sp_names = sorted({s for T in pool for s in T} | {t['p2_pokemon_state']['name'] for b in train_data for t in b['battle_timeline']})
sp_idx = {s: i for i, s in enumerate(sp_names)}

def p2_revealed(b):
    s = {t['p2_pokemon_state']['name'] for t in b['battle_timeline']}
    s.add(b['p2_lead_details']['name'])
    return s

def rating_matrices(battles):
    rT, cT, vT, rS, cS, vS = [], [], [], [], [], []
    for i, b in enumerate(battles):
        T1 = frozenset(p['name'] for p in b['p1_team_details'])
        if T1 in team_idx:
            rT.append(i); cT.append(team_idx[T1]); vT.append(1.0)
        R = p2_revealed(b)
        cand = set.intersection(*(by_species[s] for s in R)) if all(s in by_species for s in R) else set()
        if 0 < len(cand) <= 5:
            for j in cand:
                rT.append(i); cT.append(j); vT.append(-1.0 / len(cand))
        for s in T1:
            if s in sp_idx:
                rS.append(i); cS.append(sp_idx[s]); vS.append(1.0)
        for s in R:
            if s in sp_idx:
                rS.append(i); cS.append(sp_idx[s]); vS.append(-1.0)
    n = len(battles)
    return (sparse.csr_matrix((vT, (rT, cT)), shape=(n, len(pool))),
            sparse.csr_matrix((vS, (rS, cS)), shape=(n, len(sp_names))))

def bt_margin(mats_fit, y_fit, mats_apply):
    return np.column_stack([LogisticRegression(C=0.3, max_iter=2000).fit(A, y_fit).decision_function(B)
                            for A, B in zip(mats_fit, mats_apply)])

def bt_inner_oof(mats, y_fit):
    out = np.zeros((len(y_fit), len(mats)))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(len(y_fit)), y_fit):
        out[b] = bt_margin([M[a] for M in mats], y_fit[a], [M[b] for M in mats])
    return out

R_train = rating_matrices(train_data)
print(f"rating tables: {len(pool)} teams, {len(sp_names)} species")

rating tables: 4167 teams, 247 species


### 5. Model and validation

- **CatBoost only:** 5 seeds, 900 iterations, depth 4, learning rate 0.0499, l2 1.13, random strength 1.63,
  bagging temperature 0.38 (tuned in v29), on the 137 numeric columns plus the 14
  identity columns as categories (category statistics computed by CatBoost on the training part only). The 5
  probabilities are averaged; the prediction is "P1 wins" if the average is ≥ 0.5.
- The model definitions of the main blend are kept below for completeness, but only `cat_proba` is used.

Validation: 5-fold `StratifiedKFold` (seed 42). In every fold the three ratings are refitted on the training part only.
Parameters: random search of 15 configurations on split 42 (log-loss), then a paired test on 4 new splits (seeds
3/5/13/77): accuracy 0.6947 → 0.6966, better in all 4 splits, log-loss better in all 4.

In [12]:
PARAMS = {
    'XGBoost': dict(colsample_bytree=0.49242722776276354, gamma=4.847923138822793, learning_rate=0.04766754775738013,
                    max_depth=4, min_child_weight=10, n_estimators=569, reg_alpha=0.8111253665497066,
                    reg_lambda=0.11965225080478553, subsample=0.8281775897621597),
    'LightGBM': dict(colsample_bytree=0.3184434736772664, learning_rate=0.026672712582951435, min_child_samples=31,
                     n_estimators=619, num_leaves=9, reg_alpha=0.0013029309660734247, reg_lambda=4.899395467149149,
                     subsample=0.8537405378805455, subsample_freq=1),
    'HistGB': dict(max_iter=1071, learning_rate=0.023141901380918487, max_leaf_nodes=8, min_samples_leaf=31,
                   l2_regularization=0.8863050914746995, max_features=0.808611100100608),
}

def make_model(name, seed):
    if name == 'XGBoost':
        return XGBClassifier(eval_metric='logloss', n_jobs=4, random_state=seed, **PARAMS[name])
    if name == 'LightGBM':
        return LGBMClassifier(n_jobs=4, verbosity=-1, random_state=seed, **PARAMS[name])
    return HistGradientBoostingClassifier(random_state=seed, **PARAMS[name])

def ensemble_proba(X_fit, y_fit, X_apply):
    return np.mean([make_model(n, s).fit(X_fit, y_fit).predict_proba(X_apply)[:, 1] for n in PARAMS for s in SEEDS], axis=0)

import catboost
from catboost import CatBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
N_THREADS = 4
print(f"catboost {catboost.__version__}")

CAT_PARAMS = dict(iterations=900, learning_rate=0.0499, depth=4, l2_leaf_reg=1.13,
                  random_strength=1.63, bagging_temperature=0.38)   # tuned in v29

def catboost_proba(X_fit, C_fit, y_fit, X_apply, C_apply):
    names = [f'f{i}' for i in range(X_fit.shape[1])]
    fit_df = pd.concat([pd.DataFrame(X_fit, columns=names), C_fit.reset_index(drop=True)], axis=1)
    app_df = pd.concat([pd.DataFrame(X_apply, columns=names), C_apply.reset_index(drop=True)], axis=1)
    return np.mean([CatBoostClassifier(**CAT_PARAMS, random_seed=s, thread_count=N_THREADS, verbose=0,
                                       cat_features=list(C_fit.columns), allow_writing_files=False)
                    .fit(fit_df, y_fit).predict_proba(app_df)[:, 1] for s in SEEDS], axis=0)

def logistic_proba(X_fit, y_fit, X_apply):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000))
    return model.fit(np.nan_to_num(X_fit), y_fit).predict_proba(np.nan_to_num(X_apply))[:, 1]

catboost 1.2.10


In [13]:
DU = duel_train.values

def final_proba(X_fit, C_fit, y_fit, X_apply, C_apply):
    return catboost_proba(X_fit, C_fit, y_fit, X_apply, C_apply)      # CatBoost only

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(y))
for fold, (tr, va) in enumerate(cv.split(X_train, y), 1):
    t = time.time()
    R_tr, R_va = [M[tr] for M in R_train], [M[va] for M in R_train]
    bt_tr = bt_inner_oof(R_tr, y[tr])
    bt_va = bt_margin(R_tr, y[tr], R_va)
    tk_tr, tk_va = tok_inner_oof(TM_train[tr], y[tr]), tok_margin(TM_train[tr], y[tr], TM_train[va])
    X_fit = np.hstack([X_train[tr], bt_tr, DU[tr], tk_tr[:, None], X_p2[tr]])
    X_val = np.hstack([X_train[va], bt_va, DU[va], tk_va[:, None], X_p2[va]])
    oof[va] = final_proba(X_fit, C_plus.iloc[tr], y[tr], X_val, C_plus.iloc[va])
    print(f"fold {fold}: accuracy {accuracy_score(y[va], oof[va] >= 0.5):.4f} | AUC {roc_auc_score(y[va], oof[va]):.4f} ({time.time() - t:.0f}s)")

print(f"\n5-fold CV: accuracy {accuracy_score(y, oof >= 0.5):.4f} | AUC {roc_auc_score(y, oof):.4f}")

fold 1: accuracy 0.6975 | AUC 0.7730 (98s)
fold 2: accuracy 0.6970 | AUC 0.7692 (99s)
fold 3: accuracy 0.6875 | AUC 0.7658 (103s)
fold 4: accuracy 0.7100 | AUC 0.7851 (107s)
fold 5: accuracy 0.6920 | AUC 0.7714 (98s)

5-fold CV: accuracy 0.6968 | AUC 0.7730


### 6. Final training and test predictions

- Ratings (teams, species, tokens) for the training rows: inner 5-fold, as in validation. For the test rows: models
  fitted on all of train.
- The 5 CatBoost models are trained on all 10,000 training battles.
- `test.jsonl` is loaded only now and used only to compute features and predictions. Its `battle_id`s (0-4999)
  coincide with ids of *different* training battles, so for the test the move table is used as is, without removing
  any battle.
- The submission follows the order of `sample_submission.csv`, as in the starter notebook.

In [14]:
t = time.time()
bt_train = bt_inner_oof(list(R_train), y)
tk_train = tok_inner_oof(TM_train, y)
model_full_X = np.hstack([X_train, bt_train, DU, tk_train[:, None], X_p2])

with open(DATA_DIR / 'test.jsonl') as f:
    test_data = [json.loads(line) for line in f]
assert all('player_won' not in b for b in test_data)

records_test = (species_records[0], {})            # no leave-one-out on test (ids overlap different train battles)
test_df = pd.DataFrame([build_feature_row(b, records_test) for b in test_data]).fillna(0)
X_test = test_df.reindex(columns=feature_cols, fill_value=0).values.astype(float)
R_test = rating_matrices(test_data)
bt_test = bt_margin(list(R_train), y, list(R_test))
DU_test = pd.DataFrame([extract_duels(b) for b in test_data])[duel_train.columns].values
tk_test = tok_margin(TM_train, y, token_matrix([battle_tokens(b) for b in test_data]))

X_p2_test = pd.DataFrame([extract_p2_identified(b) for b in test_data])[P2_TEAM_COLS].values.astype(float)
C_test = pd.concat([pd.DataFrame([categorical_row(b) for b in test_data])[CAT_COLS],
                    pd.DataFrame([extra_categorical_row(b) for b in test_data])[EXTRA_COLS]], axis=1)

p_test = final_proba(model_full_X, C_plus, y, np.hstack([X_test, bt_test, DU_test, tk_test[:, None], X_p2_test]), C_test)

sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert list(sample_submission.columns) == ['battle_id', 'player_won']
pred_by_id = pd.Series((p_test >= 0.5).astype(int), index=test_df['battle_id'].astype(int))
submission = sample_submission.copy()
submission['player_won'] = submission['battle_id'].map(pred_by_id)
assert submission['player_won'].notna().all() and len(submission) == len(test_data)
submission['player_won'] = submission['player_won'].astype(int)
submission.to_csv('submission.csv', index=False)

print(f"submission.csv written: {len(submission)} rows, predicted P1 win rate {submission.player_won.mean():.3f} ({time.time() - t:.0f}s)")
submission.head()

submission.csv written: 5000 rows, predicted P1 win rate 0.501 (139s)


,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1


### 7. Submitting

*Save Version → Save & Run All (Commit)*, then *Submit to Competition* from the notebook viewer: the submitted file is the
`submission.csv` produced by this run.